In [5]:
import pandas as pd

df = pd.read_csv("clean_data/clean_train.csv")
df.head()

,id,article,highlights
0,0001d1afc246a7964130f43ae940af6bc6c57f01,By . Associated Press . PUBLISHED: . 14:11 EST...,"Bishop John Folda, of North Dakota, is taking ..."
1,0002095e55fcbd3a2f366d9bf92a95433dc305ef,(CNN) -- Ralph Mata was an internal affairs li...,Criminal complaint: Cop used his role to help ...
2,00027e965c8264c35cc1bc55556db388da82b07f,A drunk driver who killed a young woman in a h...,"Craig Eccleston-Todd, 27, had drunk at least t..."
3,0002c17436637c4fe1837c935c04de47adb18e9a,(CNN) -- With a breezy sweep of his pen Presid...,Nina dos Santos says Europe must be ready to a...
4,0003ad6ef0c37534f80b55b4235108024b407f0b,Fleetwood are the only team still to have a 10...,Fleetwood top of League One after 2-0 win at S...


In [6]:
import re
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))

In [24]:
def fast_sentence_split(text):
    if not isinstance(text, str):
        return []
    return [s.strip() for s in text.split('.') if len(s.strip()) > 5]

In [25]:
def preprocess_text_fast(text):
    if not isinstance(text, str):
        return {"sentences": [], "tokens": [], "clean_text": ""}
    
    # clean first
    clean_text = text.lower()
    clean_text = re.sub(r'[^a-zA-Z\s]', '', clean_text)
    
    # fast sentence split (original text)
    sentences = fast_sentence_split(text)
    sentences = [s.strip() for s in sentences if len(s.split()) > 5]
    
    # tokenize cleaned text
    words = word_tokenize(clean_text)
    
    # remove stopwords
    words = [w for w in words if w not in stop_words and w.strip() != ""]
    
    return {
        "sentences": sentences,
        "tokens": words,
        "clean_text": " ".join(words)
    }

In [26]:
sample = df["article"].iloc[0]

result = preprocess_text_fast(sample)

print(result["sentences"][:3])
print(result["clean_text"][:200])

["Animal lover Bill Oddie has become the latest victim of the plague of so-called 'super rats' sweeping Britain", 'The BBC presenter revealed more than 30 of the creatures invaded the garden of his home in Hampstead, north London earlier this week', 'Despite being known for hosting wildlife programmes and recently condemning the killing of migrating birds, the 72-year-old called in pest controllers to get rid the creatures']
richard spillett animal lover bill oddie become latest victim plague socalled super rats sweeping britain bbc presenter revealed creatures invaded garden home hampstead north london earlier week despi


In [27]:
from tqdm import tqdm
tqdm.pandas()

df = df.sample(5000, random_state=42)

df["processed"] = df["article"].progress_apply(preprocess_text_fast)

100%|██████████| 5000/5000 [00:15<00:00, 320.64it/s]


In [28]:

df["sentences"] = df["processed"].apply(lambda x: x["sentences"])
df["clean_text"] = df["processed"].apply(lambda x: x["clean_text"])

In [29]:
df.to_csv("clean_data/preprocessed_train_fast.csv", index=False)

In [30]:
print(df.shape)
print(df.columns)
print(df.head(1))

(5000, 6)
Index(['id', 'article', 'highlights', 'processed', 'sentences', 'clean_text'], dtype='str')
                                              id  \
167391  647f594b35ed395fea1cb34a99f13b741dec0d78   

                                                  article  \
167391  London (CNN) -- From Selfridges in London to F...   

                                               highlights  \
167391  Chinese designers raising profiles in the trad...   

                                                processed  \
167391  {'sentences': ['London (CNN) -- From Selfridge...   

                                                sentences  \
167391  [London (CNN) -- From Selfridges in London to ...   

                                               clean_text  
167391  london cnn selfridges london fifth avenue new ...  


In [31]:
print(df["processed"].iloc[0])

{'sentences': ["London (CNN) -- From Selfridges in London to Fifth Avenue in New York', the sight of Chinese shoppers flocking to luxury clothing stores is now a familiar one for many in the West", 'With its seemingly insatiable appetite for European and American luxury brands, China accounts for more than a quarter of the global luxury market -- and that number is growing, according to market analysts like McKinsey & Co', "But the rising fashion credentials of the country's consumers is not the only reason people are talking about China on the sidelines of this season's runway shows in London and Paris", 'A new, young generation of Chinese fashion designers is causing a stir within stylish circles', "Read more: Will Chinese designers get left behind in China's fashion boom? As Paris Fashion Week begins Tuesday, marking the culmination of the fashion world's twice-yearly dash between New York, London, Milan and Paris, these Chinese designers are yet again raising their profile in the t

In [32]:
sample = df["article"].iloc[0]
result = preprocess_text_fast(sample)

print("Original:\n", sample[:300])
print("\nSentences:\n", result["sentences"][:3])
print("\nClean:\n", result["clean_text"][:200])

Original:
 London (CNN) -- From Selfridges in London to Fifth Avenue in New York', the sight of Chinese shoppers flocking to luxury clothing stores is now a familiar one for many in the West. With its seemingly insatiable appetite for European and American luxury brands, China accounts for more than a quarter 

Sentences:
 ["London (CNN) -- From Selfridges in London to Fifth Avenue in New York', the sight of Chinese shoppers flocking to luxury clothing stores is now a familiar one for many in the West", 'With its seemingly insatiable appetite for European and American luxury brands, China accounts for more than a quarter of the global luxury market -- and that number is growing, according to market analysts like McKinsey & Co', "But the rising fashion credentials of the country's consumers is not the only reason people are talking about China on the sidelines of this season's runway shows in London and Paris"]

Clean:
 london cnn selfridges london fifth avenue new york sight chinese sh

In [33]:
print(df["processed"].isnull().sum())

0


In [34]:
df.sample(3)[["sentences", "clean_text"]]

,sentences,clean_text
30482,"[Paul Massara, CEO of Npower, is sending a let...",victoria bischoff published est december updat...
62404,"[At first glance, this psychedelic satellite i...",sarah griffiths published est august updated e...
128858,"[He's famous for his casual style, but Faceboo...",daily mail reporter published est march update...
